# Etapa 2 — Preparar a estrutura do projeto
Execute todas as células na pasta Git do Databricks, na branch `chore/estrutura-inicial`.

Esta etapa importa o pacote Python, valida parâmetros e cria/verifica cinco schemas dev. Não cria tabelas nem volumes. O path mostrado é planejado; acesso a arquivos será validado na próxima etapa. Não é necessário instalar bibliotecas.


In [ ]:
from pathlib import Path
import sys

atual = Path.cwd().resolve()
candidatos = [atual, *atual.parents]
raiz = next(
    (p for p in candidatos if (p / "src" / "medallion" / "__init__.py").is_file()
     and (p / "pyproject.toml").is_file()),
    None,
)
if raiz is None:
    raise RuntimeError(
        "Não encontrei a raiz do repositório. Abra este notebook dentro da pasta Git "
        "e confirme que src/medallion e pyproject.toml vieram no Pull. "
        "Se necessário, use %cd com o caminho da pasta Git e execute novamente."
    )
src = str(raiz / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from medallion.config import PipelineConfig
from medallion.bootstrap import preparar_ambiente

print("Pacote medallion importado de:", raiz)



In [ ]:
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.dropdown("modo", "incremental", ["incremental", "reprocessamento"])
dbutils.widgets.text("data_referencia", "")
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("raiz_arquivos", "")

parametros = {
    "ambiente": "dev",
    "catalogo": dbutils.widgets.get("catalogo").strip(),
    "modo": dbutils.widgets.get("modo"),
    "data_referencia": dbutils.widgets.get("data_referencia").strip() or None,
    "raiz_arquivos": dbutils.widgets.get("raiz_arquivos").strip() or None,
}
batch_id = dbutils.widgets.get("batch_id").strip()
if batch_id:
    parametros["batch_id"] = batch_id

config = PipelineConfig(**parametros)
print("Ambiente:", config.ambiente)
print("Catálogo:", config.catalogo)
print("Batch:", config.batch_id)
print("Entrada planejada (ainda não validada):", config.caminho_entrada)



In [ ]:
schemas = preparar_ambiente(spark, config)

print("ESTRUTURA DO PROJETO: OK")
print("Pacote Python: importado")
print("Configuração: validada")
print(f"Ambiente: {config.ambiente}")
print(f"Catálogo: {config.catalogo}")
for schema in schemas:
    print(f"Schema disponível: {schema}")
print("Próxima etapa: contratos, Volume e dados sintéticos.")

